## 0. INSTALL REQUIRED LIBRARIES

In [1]:
!pip install -q transformers accelerate sentencepiece
!pip install -q peft

## 1️. IMPORT LIBRARIES & CHECK GPU

In [2]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch version: 2.9.0+cu126
CUDA available: True


## 2. LOAD TRAINING & TEST DATA

In [3]:
train_path = "/kaggle/input/datasets/danielgodwin/nvidia-nemotron-model-reasoning-challenge/train.csv"
test_path = "/kaggle/input/datasets/danielgodwin/nvidia-nemotron-model-reasoning-challenge/test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample train row:")
print(train.iloc[0])

Train shape: (9500, 3)
Test shape: (3, 2)
Sample train row:
id                                                 00066667
prompt    In Alice's Wonderland, a secret bit manipulati...
answer                                             10010111
Name: 0, dtype: object


## 3. PREPARE DATA FOR FINE-TUNING
### Convert to instruction style

In [4]:
train_data = [
    {
        "input_text": f"Input binary: {row['prompt'].split(':')[-1].strip()}\nOutput binary:",
        "target_text": row['answer']
    }
    for _, row in train.iterrows()
]

# Convert to HuggingFace Dataset format
from datasets import Dataset
hf_train_dataset = Dataset.from_list(train_data)

print("ran succesfull")

ran succesfull


# 4️. LOAD MODEL AND APPLY LoRA

In [5]:
model_name_or_path = "mistralai/Mistral-7B-Instruct-v0.1"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=True)

# Model
model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Apply LoRA for parameter-efficient fine-tuning
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # Mistral 7B attention modules
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Silence weight tying warnings
model.config.tie_word_embeddings = False

print("model ran succesful")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

model ran succesful


## 5. LOAD TOKENIZER & MODEL

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name_or_path = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    device_map="auto",
    torch_dtype=torch.float16
)

# Ensure the tokenizer has a pad token

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# TOKENIZE DATA

def tokenize_fn(examples):
    inputs = examples["input_text"]
    targets = examples["target_text"]
    model_inputs = tokenizer(
        inputs, padding="max_length", truncation=True, max_length=50
    )
    labels = tokenizer(
        targets, padding="max_length", truncation=True, max_length=8
    ).input_ids
    model_inputs["labels"] = labels
    return model_inputs

hf_train_dataset = hf_train_dataset.map(tokenize_fn, batched=True)

from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True)

print("✅ Tokenize data ran successfully")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Map:   0%|          | 0/9500 [00:00<?, ? examples/s]

✅ Tokenize data ran successfully


## 6. TRAINING CONFIGURATION WITH TRANSFORMERS COMPATIBLE

In [7]:
from transformers import Trainer, TrainingArguments

# Freeze base model weights except LoRA layers
for name, param in model.named_parameters():
    if "lora" not in name.lower():
        param.requires_grad = False

print("✅ Base model frozen, only LoRA layers will train")

# Compatible TrainingArguments
training_args = TrainingArguments(
    output_dir="./lora_bitmodel",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    num_train_epochs=3,
    fp16=True,                 # mixed precision
    save_strategy="epoch",
    logging_steps=50,
    report_to="none"
)

# Trainer (old version compatible)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train_dataset,
    data_collator=data_collator
)

print("✅ Trainer ready. You can start training with trainer.train()")

✅ Base model frozen, only LoRA layers will train
✅ Trainer ready. You can start training with trainer.train()


## 7. BUILDING PROMPTS

In [8]:
def build_prompt(input_bin):
    return f"Input binary: {input_bin}\nOutput binary:"
print("done")

done


## 8. Make predictions

In [9]:
def predict_bit(input_bin):
    prompt = build_prompt(input_bin)
    
    # Tokenize & move to device
    tokens = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)
    
    # Generate output safely
    with torch.no_grad():
        output_tokens = model.generate(
            **tokens,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    
    # Decode and extract output
    output_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    result = output_text.split("Output binary:")[-1].strip()
    
    # Ensure 8-bit output (optional)
    if len(result) < 8:
        result = result.zfill(8)
    return result
print("done")

done


## 9. SAMPLE INPUT AND PREDICTION OUTPUT

In [10]:
sample_input = "00110100"
predicted_output = predict_bit(sample_input)
print(f"sample input:{sample_input}")
print("Predicted output:", predicted_output)

sample input:00110100
Predicted output: 00011010


## 10. BATCH INFERENCE HELPER

In [11]:
import pandas as pd
from tqdm import tqdm

def batch_predict(test_df, input_col="prompt", output_col="predicted"):
    """
    Predicts outputs for a dataframe of inputs using the fine-tuned model.
    
    Args:
        test_df (pd.DataFrame): DataFrame with input column
        input_col (str): Name of column containing input binary strings
        output_col (str): Name of column to store predictions
        
    Returns:
        pd.DataFrame: Original DataFrame with predictions column added
    """
    predictions = []
    for inp in tqdm(test_df[input_col], desc="Predicting"):
        pred = predict_bit(inp)
        predictions.append(pred)
    test_df[output_col] = predictions
    return test_df
print("done")

done


## 11. RUN BATCH INFERENCE HELPER

In [12]:
# Run batch inference on your test set
test_with_preds = batch_predict(test, input_col="prompt", output_col="predicted")

# Show sample results
print(test_with_preds.head())

# Optionally, save to CSV for submission
test_with_preds.to_csv("submission.csv", index=False)

Predicting: 100%|██████████| 3/3 [00:02<00:00,  1.45it/s]

         id                                             prompt predicted
0  00066667  In Alice's Wonderland, a secret bit manipulati...  00000010
1  000b53cf  In Alice's Wonderland, a secret bit manipulati...  00000000
2  00189f6a  In Alice's Wonderland, secret encryption rules...  00100000


## 12. SUBMITTION

In [13]:
# columns: id and prediction
submission = test_with_preds[["id", "predicted"]]

# Save to CSV
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("✅ submission.csv created!")

assert submission["predicted"].apply(lambda x: len(x) == 8 and set(x) <= {"0","1"}).all(), "Check failed: Not all outputs are 8-bit binaries!"
print("✅ All predictions are valid 8-bit binaries.")

✅ submission.csv created!
✅ All predictions are valid 8-bit binaries.
